In [1]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery

In [2]:
load_dotenv()

True

In [5]:
project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
bucket_name = os.getenv("BUCKET_NAME")
cleaned_folder = os.getenv("CLEANED_FOLDER_NAME")

gs_folder = f"gs://{bucket_name}/{cleaned_folder}/date=2026-04-11/*.parquet"

In [15]:
hive_partitioning_options = bigquery.HivePartitioningOptions()
hive_partitioning_options.mode="AUTO"
hive_partitioning_options.source_uri_prefix=f"gs://{bucket_name}/{cleaned_folder}/"


job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        autodetect=True,
        write_disposition="WRITE_TRUNCATE",
        hive_partitioning=hive_partitioning_options
    )

In [16]:
def upload_to_bq(uri, project_id, dataset, job_config):
    client = bigquery.Client()

    staging_table_id = f"{project_id}.{dataset}.test_load_staging"

    load_job = client.load_table_from_uri(
        uri, staging_table_id, job_config=job_config
    )
    
    # Waits for the job to complete.
    load_job.result()

    # Get the staging table to check the number of rows loaded
    staging_table = client.get_table(staging_table_id)
    print("Loaded {} rows.".format(staging_table.num_rows))


In [17]:
upload_to_bq(gs_folder, project_id, dataset, job_config)

Loaded 29 rows.


In [1]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from google.cloud import bigquery
from google.cloud import storage

In [2]:
load_dotenv()

True

In [34]:
project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
bucket_name = os.getenv('BUCKET_NAME')
clean_folder = os.getenv('CLEANED_FOLDER_NAME')

In [35]:
gcs_client = storage.Client()
bucket = gcs_client.bucket(bucket_name)

In [66]:
filepath = f"gs://{bucket_name}/{clean_folder}/date=2026-04-11/*.parquet"
# filepath = f"gs://{bucket_name}/{clean_folder}/date=2026-04-11/"
# filepath = f"gs://{bucket_name}/{clean_folder}/date=2026-04-11/part-00000-132b1d3c-f538-4a66-b9f0-e070f99fab1d.c000.snappy.parquet"
# filepath = f"{clean_folder}/date=2026-04-11/"
# filepath = f"{clean_folder}/date=2026-04-11/part-00000-132b1d3c-f538-4a66-b9f0-e070f99fab1d.c000.snappy.parquet"


In [67]:
filepath

'gs://mnl_accident_pipeline_bucket/cleaned_v2/date=2026-04-11/*.parquet'

In [68]:
path = filepath.replace("gs://", "")
path

'mnl_accident_pipeline_bucket/cleaned_v2/date=2026-04-11/*.parquet'

In [69]:
bucket_name, blob_path = path.split("/", 1)
bucket_name, blob_path

('mnl_accident_pipeline_bucket', 'cleaned_v2/date=2026-04-11/*.parquet')

In [70]:
prefix = blob_path.split("*")[0]
prefix

'cleaned_v2/date=2026-04-11/'

In [ ]:
bucket = gcs_client.bucket(bucket_name)
blobs = gcs_client.list_blobs(bucket, prefix=prefix)


In [72]:
for blob in blobs:
    print(blob.name)

cleaned_v2/date=2026-04-11/part-00000-132b1d3c-f538-4a66-b9f0-e070f99fab1d.c000.snappy.parquet


In [ ]:
def gcs_path_exists(gcs_client, uri):
    # Remove "gs://"
    path = uri.replace("gs://", "")
    bucket_name, blob_path = path.split("/", 1)

    # Extract prefix before wildcard
    prefix = blob_path.split("*")[0]

    bucket = gcs_client.bucket(bucket_name)
    blobs = gcs_client.list_blobs(bucket, prefix=prefix)

    # Check if any blob matches full pattern
    for blob in blobs:
        if blob.name.endswith(".parquet"):
            return True

    return False